# 00 — Prepare the sample data

The official feature store notebooks read sample data from
`wasbs://data@azuremlexampledata.blob.core.windows.net/feature-store-prp/`. That storage account now rejects
anonymous reads with `PublicAccessNotPermitted`, so the official notebooks fail at the first data read
([azureml-examples#3738](https://github.com/Azure/azureml-examples/issues/3738)).

This notebook is the only new code in the module. It generates synthetic data with the **same paths, columns,
and types** that the official notebooks, transformation code, and training and scoring scripts expect, then:

- **[Local]** writes a copy under `workshop/outputs/feature_store_sample_data/` for local Spark development
- **[Control plane]** uploads the same files to the Spark ADLS Gen2 account used by registered feature sets,
  materialization jobs, and pipelines

| Dataset | Columns | Used by |
|---|---|---|
| `datasources/transactions-source` | `transactionID`, `accountID`, `timestamp`, `transactionAmount` | `transactions` feature set (3-day and 7-day rolling aggregates) |
| `datasources/accounts-precalculated` | `accountID`, `timestamp`, `accountCountry`, `isUserRegistered`, `numPaymentRejects1dPerUser`, `accountAge` | `accounts` feature set (precomputed, yearly snapshots) |
| `observation_data/train` | `transactionID`, `accountID`, `timestamp`, `transactionAmount`, `localHour`, `transactionAmountUSD`, `digitalItemCount`, `physicalItemCount`, `isProxyIP`, `is_fraud` | Training data (point-in-time join) |
| `observation_data/batch_inference` | Training columns without `is_fraud` | Batch inference pipeline |

Transactions continue three days past today so online materialization has fresh values during the demo:
online values older than a feature set's `temporal_join_lookback` (one day for `transactions`) aren't kept.
The account IDs used by the official online lookup and endpoint test samples are included, with a transaction
every two hours from two days ago until the end of the data.

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from azure.identity import AzureCliCredential
from azure.storage.filedatalake import DataLakeServiceClient
from dotenv import load_dotenv

load_dotenv(Path.cwd().parents[1] / ".env", override=True)
local_root = Path.cwd().parents[1] / "outputs" / "feature_store_sample_data" / "feature-store-prp"
storage_account = os.environ["FEATURE_STORE_SOURCE_STORAGE_ACCOUNT"]
filesystem = os.environ["FEATURE_STORE_SOURCE_FILESYSTEM"]
rng = np.random.default_rng(42)
print(f"Local copy: {local_root}")
print(f"Cloud copy: abfss://{filesystem}@{storage_account}.dfs.core.windows.net/feature-store-prp")

### **[Local]** Accounts and transactions

In [ ]:
# Account IDs used by the official online lookup and online endpoint samples.
sample_accounts = ["A985156952816816", "A1055521248929430", "A914800935560176", "A1055521510328860"]
generated = {f"A{number}" for number in rng.integers(10**14, 10**16, size=600)} - set(sample_accounts)
account_ids = np.array(sample_accounts + sorted(generated)[:496])
n_accounts = len(account_ids)

created = pd.Timestamp("2015-01-01", tz="UTC") + pd.to_timedelta(rng.integers(0, 7 * 365, n_accounts), unit="D")
risk = rng.beta(2, 8, n_accounts)
country = rng.choice(["US", "GB", "CA", "DE", "IN", "BR"], n_accounts, p=[0.45, 0.12, 0.1, 0.1, 0.13, 0.1])
registered = rng.random(n_accounts) < 0.8
daily_rate = rng.uniform(0.2, 1.0, n_accounts)
daily_rate[: len(sample_accounts)] = 3.0

# Account profiles are precomputed once a year, as in the official accounts feature set.
accounts = pd.concat(
    pd.DataFrame({
        "accountID": account_ids,
        "timestamp": pd.Timestamp(f"{year}-01-01", tz="UTC"),
        "accountCountry": country,
        "isUserRegistered": registered,
        "numPaymentRejects1dPerUser": rng.poisson(0.5 + 6 * risk).astype("float64"),
        "accountAge": (pd.Timestamp(f"{year}-01-01", tz="UTC") - created).days.astype("float64"),
    })
    for year in range(2022, 2027)
)

start = pd.Timestamp("2022-01-01", tz="UTC")
end = (pd.Timestamp.now(tz="UTC") + pd.Timedelta(days=3)).normalize()
days = (end - start).days
counts = rng.poisson(daily_rate[:, None], size=(n_accounts, days)).ravel()
account_index = np.repeat(np.repeat(np.arange(n_accounts), days), counts)
day_index = np.repeat(np.tile(np.arange(days), n_accounts), counts)
risky = rng.random(len(account_index)) < risk[account_index] * 0.2
transactions = pd.DataFrame({
    "transactionID": [f"T{number:09d}" for number in range(len(account_index))],
    "accountID": account_ids[account_index],
    "timestamp": start
    + pd.to_timedelta(day_index, unit="D")
    + pd.to_timedelta(rng.integers(0, 86400, len(account_index)), unit="s"),
    "transactionAmount": np.round(rng.lognormal(np.log(40), 0.9, len(account_index)) * (1 + 2 * risky), 2),
}).sort_values("timestamp", ignore_index=True)
print(f"{n_accounts} accounts, {len(accounts)} account snapshots, {len(transactions):,} transactions")
print(f"Transactions from {transactions.timestamp.min()} to {transactions.timestamp.max()}")

### **[Local]** Observation data for training and batch inference

In [ ]:
def observations(frame: pd.DataFrame, rows: int, labeled: bool) -> pd.DataFrame:
    sample = frame.sample(n=rows, random_state=42).sort_values("timestamp", ignore_index=True)
    index = pd.Index(account_ids).get_indexer(sample["accountID"])
    year = sample["timestamp"].dt.year.clip(upper=2026)
    rejects = accounts.set_index(["accountID", accounts["timestamp"].dt.year])["numPaymentRejects1dPerUser"]
    proxy = rng.random(rows) < 0.05 + 0.4 * risk[index]
    result = sample.assign(
        localHour=sample["timestamp"].dt.hour,
        transactionAmountUSD=np.round(sample["transactionAmount"] * rng.uniform(0.8, 1.2, rows), 2),
        digitalItemCount=rng.poisson(1.2, rows),
        physicalItemCount=rng.poisson(0.8, rows),
        isProxyIP=proxy,
    )
    if labeled:
        score = (
            -5.5
            + 7 * risk[index]
            + 1.5 * (sample["transactionAmount"].to_numpy() > 150)
            + 1.0 * proxy
            + 0.2 * rejects.reindex(list(zip(sample["accountID"], year))).to_numpy()
        )
        result["is_fraud"] = rng.random(rows) < 1 / (1 + np.exp(-score))
    return result

in_window = lambda first, last: transactions[
    (transactions["timestamp"] >= pd.Timestamp(first, tz="UTC"))
    & (transactions["timestamp"] < pd.Timestamp(last, tz="UTC"))
]
train = observations(in_window("2023-01-08", "2023-06-29"), 20000, labeled=True)
batch_inference = observations(in_window("2023-06-01", "2023-06-29"), 5000, labeled=False)
print(f"Training observations: {len(train):,}; fraud rate {train['is_fraud'].mean():.1%}")
print(f"Batch inference observations: {len(batch_inference):,}")
display(train.head())

### **[Local]** Write the local copy, then **[Control plane]** upload it to ADLS Gen2

In [ ]:
# Online values older than temporal_join_lookback (one day for transactions) aren't kept, so give the
# sample accounts used by the online lookup and endpoint test a transaction every two hours around the demo.
demo_times = pd.date_range(
    pd.Timestamp.now(tz="UTC").floor("h") - pd.Timedelta(days=2), end - pd.Timedelta(hours=1), freq="2h"
)
demo_activity = pd.DataFrame({
    "transactionID": [f"D{number:09d}" for number in range(len(demo_times) * len(sample_accounts))],
    "accountID": np.repeat(sample_accounts, len(demo_times)),
    "timestamp": np.tile(demo_times, len(sample_accounts)),
    "transactionAmount": np.round(np.random.default_rng(7).lognormal(np.log(40), 0.5, len(demo_times) * len(sample_accounts)), 2),
})
print(f"{len(demo_activity)} demo transactions from {demo_times.min()} to {demo_times.max()}")

In [ ]:
datasets = {
    "datasources/transactions-source/part-00000.parquet": transactions,
    "datasources/transactions-source/part-00001.parquet": demo_activity,
    "datasources/accounts-precalculated/part-00000.parquet": accounts,
    "observation_data/train/part-00000.parquet": train,
    "observation_data/batch_inference/part-00000.parquet": batch_inference,
}

service = DataLakeServiceClient(
    account_url=f"https://{storage_account}.dfs.core.windows.net", credential=AzureCliCredential(process_timeout=60)
)
file_system = service.get_file_system_client(filesystem)
for relative_path, frame in datasets.items():
    local_file = local_root / relative_path
    local_file.parent.mkdir(parents=True, exist_ok=True)
    # Spark 3.5 reads microsecond timestamps.
    pq.write_table(
        pa.Table.from_pandas(frame, preserve_index=False),
        local_file,
        coerce_timestamps="us",
        allow_truncated_timestamps=True,
    )
    remote_path = f"feature-store-prp/{relative_path}"
    file_system.get_directory_client(str(Path(remote_path).parent.as_posix())).create_directory()
    file_system.get_file_client(remote_path).upload_data(local_file.read_bytes(), overwrite=True)
    print(f"{relative_path}: {len(frame):,} rows, {local_file.stat().st_size / 1e6:.1f} MB")

In [ ]:
uploaded = {
    path.name: path.content_length
    for path in file_system.get_paths(path="feature-store-prp", recursive=True)
    if not path.is_directory and path.name.removeprefix("feature-store-prp/") in datasets
}
for name, size in uploaded.items():
    print(f"{name}: {size / 1e6:.1f} MB")
assert len(uploaded) == len(datasets), "Every dataset must be uploaded"